In [ ]:
-- ============================================
-- Set Role for Liquidity Risk Analysis - What-If Scenarios
-- ============================================
USE ROLE LIQUIDITY_RISK_ROLE;
USE WAREHOUSE LIQUIDITY_RISK_WH;
USE DATABASE LIQUIDITY_RISK_DB;

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.query_tag = '{"origin":"sf_sit-is","name":"liquidity_risk_agent","version":{"major":1,"minor":0},"attributes":{"is_quickstart":1,"source":"notebook"}}'

import snowflake.snowpark as snowpark
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *
import datetime
import sys

import utils
import prod_calculations as pc

In [ ]:
n_days = 150

db = "LIQUIDITY_RISK_DB"
positions_table = f'{db}.RAW_SANDBOX.POSITIONS'
inflows_table = f'{db}.RAW_SANDBOX.CASH_INFLOWS'
outflows_table = f'{db}.RAW_SANDBOX.CASH_OUTFLOWS'
lcr_table = f'{db}.PRESENTATION.WHAT_IF_LCR'

args = [x for x in sys.argv if '/Users/' not in x ]
if args == [] or args == ['']:
    suffix = 'xyz'
    what_if_id = '1'
else:
    suffix = next((x.split("=")[1] for x in args if x.startswith("suffix=")), 'xyz')
    what_if_id = next((x.split("=")[1] for x in args if x.startswith("what_if_id=")), '1')

positions_table_clone = f"{positions_table}_{suffix}_CLONE"
outflows_table_clone = f"{outflows_table}_{suffix}_CLONE"
inflows_table_clone = f"{inflows_table}_{suffix}_CLONE"

what_if_args = {'db':db,
                'positions_table':positions_table,
                'positions_table_clone':positions_table_clone,
                'outflows_table':outflows_table,
                'outflows_table_clone':outflows_table_clone,
                'inflows_table':inflows_table,
                'inflows_table_clone':inflows_table_clone,
                'suffix':suffix,
                'what_if_id':what_if_id
               }

hqla_detail_table = f'{db}.RAW.HQLA_DETAIL_{suffix}'
hqla_summary_table = f'{db}.RAW.HQLA_SUMMARY_{suffix}'
outflows_projections_table = f'{db}.RAW.OUTFLOW_PROJECTIONS_{suffix}'
inflows_projections_table = f'{db}.RAW.INFLOW_PROJECTIONS_{suffix}'
cashflow_projections_table = f'{db}.RAW.NET_CASHFLOW_PROJECTIONS_{suffix}'

print()
print("Configuration:")
print(f"  DATABASE: {db}")
print()
print("What-If Parameters:")
print(f"  suffix: {suffix}")
print(f"  what_if_id: {what_if_id}")

In [ ]:
utils.clone_and_update(session, what_if_args)

In [ ]:
pc.run_hqla_projection(session, db, n_days, positions_table_clone, hqla_detail_table)
pc.run_summary_hqla(session, n_days, hqla_detail_table, hqla_summary_table)

pc.run_cashflow_projection(session, 'outflow', n_days, outflows_table_clone, outflows_projections_table);
pc.run_cashflow_projection(session, 'inflow', n_days, inflows_table_clone, inflows_projections_table);
pc.run_netting(session, n_days, inflows_projections_table, outflows_projections_table, cashflow_projections_table)

In [ ]:
session.sql(f"""
insert into {lcr_table} 
select {what_if_id},
       current_timestamp(),
       a.day_number,
       a.abs_net_position as total_net_cash_outflows,
       b.total_hqla_usd as hqla,
       hqla / total_net_cash_outflows as lcr
from {cashflow_projections_table} a
join {hqla_summary_table} b on a.day_number = b.day_number
order by day_number
"""
).collect()